In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='v6-dk-459909') # ถ้าตั้งค่าโปรเจกต์ไว้

training_points = ee.FeatureCollection('users/baiwann2548/Udon_Training_Points')

print(f"จำนวนจุดทั้งหมด: {training_points.size().getInfo()}")

# กำหนดขอบเขตอุดรธานี และโหลดภาพ Sentinel-2 เหมือนเดิม (แต่คราวนี้ทำใน Python)
aoi = ee.Geometry.Point([103.0236, 17.1650]).buffer(10000)

s2_image = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate('2023-01-01', '2023-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .median()
    .clip(aoi)
)



In [ ]:
# 1. ฟังก์ชันสำหรับเพิ่ม Spectral Indices (NDVI แยกพืช, NDWI แยกน้ำและความชื้น)
def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    return image.addBands([ndvi, ndwi])

# นำภาพ Sentinel-2 มาเพิ่มแบนด์ NDVI และ NDWI
image_with_features = add_indices(s2_image)

# กำหนด Features (Bands + Indices) ที่จะให้โมเดลใช้เรียนรู้
bands_to_use = [ 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI']

# 2. สกัดค่าพิกเซล (Extract/Sample) ลงในจุด Training Points
training_data = image_with_features.select(bands_to_use).sampleRegions(
    collection=training_points,
    properties=['class'],
    scale=10, # ความละเอียด Sentinel-2 คือ 10 เมตร
    tileScale=16
)

# 3. แบ่งชุดข้อมูล Train (80%) และ Validation/Test (20%)
# สร้างคอลัมน์ตัวเลขสุ่ม (0 ถึง 1)
training_data = training_data.randomColumn('random')

# กรองแยกชุดข้อมูล
train_set = training_data.filter(ee.Filter.lt('random', 0.8))
test_set = training_data.filter(ee.Filter.gte('random', 0.8))

print(f"จำนวนข้อมูลสำหรับ Train: {train_set.size().getInfo()}")
print(f"จำนวนข้อมูลสำหรับ Validation: {test_set.size().getInfo()}")

# 4. ฝึกสอนโมเดล Random Forest (ภารกิจที่ 2)
# ตั้งค่าจำนวนต้นไม้ (Trees) เป็น 50
rf_classifier = ee.Classifier.smileRandomForest(numberOfTrees=50).train(
    features=train_set,
    classProperty='class',
    inputProperties=bands_to_use
)

# 5. นำโมเดลไปจำแนกภาพทั้งพื้นที่ AOI
classified_image = image_with_features.classify(rf_classifier)

# 6. แสดงผลลัพธ์บนแผนที่ Interactive
Map2 = geemap.Map()
Map2.centerObject(aoi, 12)

# โชว์ภาพ True color เป็นพื้นหลัง
vis_true_color = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000, 'gamma': 1.4}
Map2.addLayer(image_with_features, vis_true_color, 'Sentinel-2 True Color')

# กำหนดสีให้แต่ละ Class
class_palette = ['0000FF', '00FFFF', '00FF00', 'FF0000']

Map2.addLayer(classified_image, {'min': 0, 'max': 3, 'palette': class_palette}, 'RF Classification')

Map2

In [ ]:
# 1. ให้โมเดลลองทายข้อมูลในชุดทดสอบ (test_set)
validated = test_set.classify(rf_classifier)

# 2. สร้าง Confusion Matrix
# ('class' คือความจริงที่เราจุดไว้, 'classification' คือสิ่งที่โมเดลทายออกมา)
error_matrix = validated.errorMatrix('class', 'classification')

# 3. แสดงผลลัพธ์การประเมิน
print('=== Confusion Matrix ===')
print(error_matrix.getInfo())

print('\n=== Overall Accuracy (ความแม่นยำรวม) ===')
# ค่าเข้าใกล้ 1.0 แปลว่าแม่นยำมาก
print(error_matrix.accuracy().getInfo())

print('\n=== Kappa Coefficient ===')
# ค่าความสอดคล้อง (ขจัดความบังเอิญทายถูก) เข้าใกล้ 1.0 ยิ่งดี
print(error_matrix.kappa().getInfo())

In [ ]:
# ดึงค่า Feature Importance จากโมเดล Random Forest
dict_importance = rf_classifier.explain().get('importance')

print('=== Feature Importance ===')
print(dict_importance.getInfo())

# (ทางเลือก) ถ้านำข้อมูลไปพล็อตกราฟแท่งใน Python ด้วย matplotlib จะทำให้รายงานดูสวยขึ้นมากครับ
import matplotlib.pyplot as plt

# ดึงชื่อ Features และค่าความสำคัญมาวาดกราฟ
features = list(dict_importance.getInfo().keys())
importance_values = list(dict_importance.getInfo().values())

plt.figure(figsize=(10, 6))
plt.barh(features, importance_values, color='skyblue')
plt.xlabel('Importance Score')
plt.ylabel('Features (Bands / Indices)')
plt.title('Random Forest Feature Importance')
plt.gca().invert_yaxis() # กลับด้านให้คะแนนมากอยู่บน
plt.show()

In [ ]:
# ==========================================
# 1. ฝึกสอนโมเดลที่ 2: Support Vector Machine (SVM)
# ==========================================
print("Training SVM Model...")
svm_classifier = ee.Classifier.libsvm().train(
    features=train_set,
    classProperty='class', # เปลี่ยนชื่อให้ตรงกับ property ของคุณถ้าจำเป็น
    inputProperties=bands_to_use
)

# นำไปทดสอบกับชุด Validation
validated_svm = test_set.classify(svm_classifier)
error_matrix_svm = validated_svm.errorMatrix('class', 'classification')

print('=== SVM Overall Accuracy ===')
print(error_matrix_svm.accuracy().getInfo())

print('=== SVM Kappa Coefficient ===')
print(error_matrix_svm.kappa().getInfo())


# ==========================================
# 2. ดึงค่า Producer's Accuracy, User's Accuracy (RF & SVM)
# ==========================================
print("\n=== Random Forest: Producer's & User's Accuracy ===")
# Producer's Accuracy (Error of Omission)
rf_pa = error_matrix.producersAccuracy().getInfo()
print(f"Producer's Accuracy (RF): \n{rf_pa}")

# User's Accuracy (Error of Commission) - ใน GEE ใช้คำว่า consumersAccuracy
rf_ua = error_matrix.consumersAccuracy().getInfo()
print(f"User's Accuracy (RF): \n{rf_ua}")

print("\n=== SVM: Producer's & User's Accuracy ===")
svm_pa = error_matrix_svm.producersAccuracy().getInfo()
print(f"Producer's Accuracy (SVM): \n{svm_pa}")

svm_ua = error_matrix_svm.consumersAccuracy().getInfo()
print(f"User's Accuracy (SVM): \n{svm_ua}")


In [ ]:
import numpy as np

# ฟังก์ชันคำนวณ F1-Score จัดการกับรูปแบบ List ของ GEE
def calculate_f1(pa_gee, ua_gee):
    # แกะค่าออกจากรูปแบบ List ของ GEE
    pa_list = [item[0] if isinstance(item, list) else item for item in pa_gee]
    ua_list = ua_gee[0] if isinstance(ua_gee[0], list) else ua_gee

    f1_scores = []
    for pa, ua in zip(pa_list, ua_list):
        if pa + ua == 0:
            f1_scores.append(0.0)
        else:
            f1_scores.append(2 * (pa * ua) / (pa + ua))
    return np.round(f1_scores, 3) # ปัดทศนิยม 3 ตำแหน่ง

# คำนวณ F1-Score ให้ทั้งสองโมเดล
rf_f1 = calculate_f1(rf_pa, rf_ua)
svm_f1 = calculate_f1(svm_pa, svm_ua)

print(f"F1-Score (Random Forest) ต่อ Class: {rf_f1}")
print(f"F1-Score (SVM) ต่อ Class:          {svm_f1}")